# Face detection on the KV260 PL

`fpga_top_nn`: AXI DMA MM2S → `yuyv2gray` → `unpack32to8` → `downscale4x` → **FaceNet v1** (FINN IP) → AXI DMA S2MM.

One webcam frame per DMA transfer: 640×480 YUYV (614 400 B) in, a 20×15 grid out (600 B, INT8, NHWC):
per 8×8-pixel cell of the 160×120 image a **score** (logit = int8 × out_scale) and a **size code**
(face size px = 2^(code × 4 + 3.5), code = int8 × out_scale).

Upload next to this notebook: `face_detect.bit`, `face_detect.hwh`, `hw_meta.json`, `facenet_verify_input.npy`, `facenet_expected_output.npy` (see the step README).
Close other notebooks that use the webcam.

In [ ]:
import json
import time
import numpy as np
import cv2
from pynq import Overlay, allocate

W, H        = 640, 480
IN_BYTES    = W * H * 2            # YUYV
GRID_H, GRID_W = 15, 20
OUT_BYTES   = GRID_H * GRID_W * 2  # 600
GUARD       = 64
S2MM_LENGTH = 0x58

meta  = json.load(open('hw_meta.json'))
SCALE = meta['out_scale']
print('FaceNet', meta['run'], 'out_scale', SCALE)

ol  = Overlay('face_detect.bit')
dma = ol.axi_dma_0
tx  = allocate(shape=(IN_BYTES,), dtype=np.uint8)
rx  = allocate(shape=(OUT_BYTES + GUARD,), dtype=np.uint8)

def fpga_grid(yuyv):
    """One YUYV frame through the PL. Returns (int8 grid (15, 20, 2), ms, bytes received)."""
    tx[:] = yuyv
    tx.flush()
    rx[:] = 0xEE
    rx.flush()
    t0 = time.perf_counter()
    dma.recvchannel.transfer(rx)          # S2MM first; it stops at tlast (600 bytes)
    dma.sendchannel.transfer(tx)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    ms = (time.perf_counter() - t0) * 1e3
    rx.invalidate()
    grid = np.array(rx[:OUT_BYTES]).view(np.int8).reshape(GRID_H, GRID_W, 2)
    return grid, ms, dma.read(S2MM_LENGTH)

In [ ]:
def detect(grid, thr=0.5, iou_max=0.3):
    """int8 grid -> list of (score, x0, y0, x1, y1) in 640x480 pixels."""
    score = 1 / (1 + np.exp(-grid[..., 0].astype(np.float32) * SCALE))
    size  = 2.0 ** (grid[..., 1].astype(np.float32) * SCALE * meta['size_log_span'] + meta['size_log_min'])
    pad   = np.pad(score, 1, constant_values=0)
    peak  = score >= np.max([pad[dy:dy + GRID_H, dx:dx + GRID_W] for dy in range(3) for dx in range(3)], axis=0)
    dets = []
    for gy, gx in zip(*np.nonzero(peak & (score > thr))):
        cx, cy, half = (gx + 0.5) * 8 * 4, (gy + 0.5) * 8 * 4, size[gy, gx] * 4 / 2   # 160x120 -> 640x480
        dets.append((float(score[gy, gx]), cx - half, cy - half, cx + half, cy + half))
    dets.sort(reverse=True)
    keep = []                                          # remove duplicates on the same face (overlap)
    for d in dets:
        if all(iou(d, k) < iou_max for k in keep):
            keep.append(d)
    return keep

def iou(a, b):
    ix = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    iy = max(0, min(a[4], b[4]) - max(a[2], b[2]))
    inter = ix * iy
    return inter / ((a[3] - a[1]) * (a[4] - a[2]) + (b[3] - b[1]) * (b[4] - b[2]) - inter)

def draw(rgb, dets):
    for s, x0, y0, x1, y1 in dets:
        cv2.rectangle(rgb, (int(x0), int(y0)), (int(x1), int(y1)), (255, 220, 0), 2)
        cv2.putText(rgb, f'{s:.2f}', (int(x0), max(12, int(y0) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 220, 0), 1)
    return rgb

## 1. Bit-exact check

`facenet_verify_input.npy` holds the 2 images FINN used to verify its RTL (seeded random 160×120 pixels, from `ml/finn_v1/input.npy`), and `facenet_expected_output.npy` the trained PyTorch model's output for them (`ml/finn_v1/expected_output.npy`).
Each image is turned into a 640×480 YUYV frame by repeating every pixel 4×4 (U = V = 128), so `downscale4x` returns exactly the original pixels. The PL must then give exactly the same 600 integers.

In [ ]:
images   = np.load('facenet_verify_input.npy').astype(np.uint8)[:, 0]   # (2, 120, 160)
expected = np.load('facenet_expected_output.npy')                      # (2, 2, 15, 20) float, NCHW
for k, img in enumerate(images):
    y = np.kron(img, np.ones((4, 4), np.uint8)).reshape(-1)            # 640x480 Y, each pixel 4x4
    yuyv = np.empty(IN_BYTES, np.uint8)
    yuyv[0::2], yuyv[1::2] = y, 128
    grid, ms, n = fpga_grid(yuyv)
    exp_int = np.rint(expected[k] / SCALE).astype(np.int8).transpose(1, 2, 0)   # -> (15, 20, 2) NHWC
    same = np.array_equal(grid, exp_int)
    guard = bool(np.all(rx[OUT_BYTES:] == 0xEE))
    print(f'image {k}: {"bit-exact" if same else "MISMATCH"}  ({int((grid != exp_int).sum())} of 600 values differ), '
          f'S2MM {n} B, guard {"ok" if guard else "OVERWRITTEN"}, {ms:.2f} ms')

## 2. One live frame

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'YUYV'))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, H)
cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
for _ in range(5):
    ok, frame = cap.read()
assert ok and frame.size == IN_BYTES, f'expected raw YUYV ({IN_BYTES} B), got {frame.size if ok else None}'

yuyv = np.asarray(frame, dtype=np.uint8).reshape(-1)
grid, ms, n = fpga_grid(yuyv)
dets = detect(grid)
print(f'{len(dets)} face(s), PL time {ms:.2f} ms, S2MM {n} B')
rgb = draw(cv2.cvtColor(yuyv.reshape(H, W, 2), cv2.COLOR_YUV2RGB_YUYV), dets)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6)); plt.imshow(rgb); plt.axis('off'); plt.show()

## 3. Live video (runs for `SECONDS`, stop the cell to end earlier)

Threshold `THR`: higher = fewer false boxes, more missed faces.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

SECONDS, THR = 30, 0.5
view, info = widgets.Image(format='jpeg', width=W, height=H), widgets.Label()
display(view, info)
n_frames, t_start, t_pl = 0, time.perf_counter(), 0.0
try:
    while time.perf_counter() - t_start < SECONDS:
        ok, frame = cap.read()
        yuyv = np.asarray(frame, dtype=np.uint8).reshape(-1)
        grid, ms, _ = fpga_grid(yuyv)
        dets = detect(grid, THR)
        rgb = draw(cv2.cvtColor(yuyv.reshape(H, W, 2), cv2.COLOR_YUV2BGR_YUYV), dets)
        view.value = cv2.imencode('.jpg', rgb)[1].tobytes()
        n_frames += 1
        t_pl += ms
        info.value = f'{n_frames / (time.perf_counter() - t_start):.1f} fps, PL {t_pl / n_frames:.1f} ms/frame, faces: {len(dets)}'
except KeyboardInterrupt:
    pass

In [ ]:
cap.release()
tx.freebuffer()
rx.freebuffer()